# Parametric fits: BazinFit, LinexpFit, VillarFit

Parametric fits extract physically motivated parameters from transient **flux** light curves.
All three operate on flux (not magnitude) — use positive flux values.

## Synthetic SN-like light curve

In [ ]:
import light_curve as licu
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(1)
t = np.linspace(-20, 100, 80)

# Bazin: f(t) = A * exp(-(t-t0)/fall) / (1 + exp(-(t-t0)/rise)) + B
A, t0, rise, fall, B = 100.0, 10.0, 5.0, 25.0, 5.0
flux = A * np.exp(-(t - t0) / fall) / (1 + np.exp(-(t - t0) / rise)) + B
flux_err = np.sqrt(flux) * 0.1
flux = flux + rng.normal(0, flux_err)
print(f'n_obs: {len(t)}, peak flux: {flux.max():.1f}')

## BazinFit

Fits a 5-parameter rising/falling exponential (Bazin 2011):

$$f(t) = A \cdot \frac{e^{-(t-t_0)/t_\text{fall}}}{1 + e^{-(t-t_0)/t_\text{rise}}} + B$$

Output: amplitude, reference time, rise time, fall time, baseline, and reduced $\chi^2$.

In [ ]:
bazin = licu.BazinFit(algorithm='mcmc-ceres')
result_b = bazin(t, flux, flux_err)
for name, val in zip(bazin.names, result_b):
    print(f'  {name:35s} = {val:.4f}')

## LinexpFit

Fits a linear-times-exponential (Karpenka 2012) — simpler rising part, useful for
core-collapse supernovae with a fast rise:

$$f(t) = A \cdot (1 + B\,(t - t_0)) \cdot e^{-(t - t_0) / t_\text{fall}} \cdot \mathbf{1}_{t > t_0} + C$$

Output: amplitude, reference time, fall time, baseline, and reduced $\chi^2$.

In [ ]:
linexp = licu.LinexpFit(algorithm='ceres')
result_l = linexp(t, flux, flux_err)
for name, val in zip(linexp.names, result_l):
    print(f'  {name:35s} = {val:.4f}')

## VillarFit

Fits the Villar 2019 function — a more flexible Gaussian+plateau model for SN classification:

$$f(t) = \frac{A + \beta\,(t - t_0)}{1 + e^{-(t - t_0)/t_\text{rise}}} \cdot
\begin{cases}
1 & t \leq t_0 + t_\text{plateau} \\
e^{-(t - t_0 - t_\text{plateau})/t_\text{fall}} & t > t_0 + t_\text{plateau}
\end{cases}
+ B$$

Output: amplitude, baseline, reference time, rise time, fall time,
plateau relative amplitude, plateau duration, and reduced $\chi^2$.

In [ ]:
villar = licu.VillarFit(algorithm='ceres')
result_v = villar(t, flux, flux_err)
for name, val in zip(villar.names, result_v):
    print(f'  {name:35s} = {val:.4f}')

## Comparing the fits

In [ ]:
t_grid = np.linspace(t.min(), t.max(), 300)

# Reconstruct model curves from fit parameters
A_b, t0_b, rise_b, fall_b, B_b = result_b[:5]
flux_bazin = A_b * np.exp(-(t_grid - t0_b) / fall_b) / (1 + np.exp(-(t_grid - t0_b) / rise_b)) + B_b

fig, ax = plt.subplots(figsize=(8, 4))
ax.errorbar(t, flux, yerr=flux_err, fmt='.', color='gray', alpha=0.5, label='data')
ax.plot(t_grid, flux_bazin, label=f'BazinFit  (χ²={result_b[-1]:.2f})')
ax.set_xlabel('Time (days)')
ax.set_ylabel('Flux')
ax.legend()
ax.set_title('Parametric fit example')
plt.tight_layout()
plt.show()

## Notes

- All three features operate on **flux** light curves (not magnitude).
- Available solvers: `'ceres'` (fast, gradient-based), `'mcmc-ceres'` (MCMC warm-start + Ceres refinement), `'lmsder'` (requires GSL).
- For multi-band transient fitting with a physical blackbody model, see the
  [RainbowFit tutorial](multiband/tutorial.ipynb).
- [API reference](api/fitting.md)